# Does the GRU rediscover HAR?

HAR uses the average range over the last 1, 5 and 22 days. The GRU reads only 20 days, so here we use 1, 5 and 20.

- **Data:** the development test years 2017-2022. The holdout isn't used.
- **Models:** the step 7 GRUs, refitted with their saved settings. The code checks that they reproduce the saved forecasts.
- **Scores:** every line is fitted on earlier years and scored on the test year.

Four questions:
1. Does the GRU's memory hold the averages, and more than an untrained GRU's?
2. Is the GRU's forecast just a straight line on the averages?
3. Does training the GRU help at all?
4. If we erase an average from the memory, does the forecast get worse?

In [1]:
from pathlib import Path

import pandas as pd
from joblib import Parallel, delayed
from sklearn.linear_model import LinearRegression

from data import load_all
from gru_probes import WINDOWS, probe_market, r2, r2_vs_average

markets = load_all()
SEEDS = [0, 1, 2]
AVERAGES = [f'avg_{n}' for n in WINDOWS]
RESULTS = Path('results')

def run_one(market, seed):
    """Load the saved probes, or run them and save them."""
    path = RESULTS / f'probes_{market}_seed{seed}.pkl'
    if path.exists():
        return pd.read_pickle(path)
    preds = probe_market(markets[market], pd.read_pickle(RESULTS / f'gru_seed{seed}_{market}.pkl'), seed)
    preds.to_pickle(path)
    return preds

keys = [(market, seed) for seed in SEEDS for market in markets]
runs = dict(zip(keys, Parallel(n_jobs=3)(delayed(run_one)(market, seed) for market, seed in keys)))
har = {market: pd.read_pickle(RESULTS / f'har_{market}.pkl') for market in markets}

def per_market(score):
    """Average one score over the three seeds, for each market."""
    return {market: sum(score(runs[market, seed]) for seed in SEEDS) / len(SEEDS) for market in markets}

## 1. Does the memory hold the averages?

R2 of a straight line from memory to each average, on the test years. **Trained** is the real GRU. **Random** is the same GRU before any training.

In [2]:
table = {}
for kind in ['trained', 'random']:
    for avg in AVERAGES:
        table[kind, avg] = per_market(lambda p: r2(p[f'{kind}_{avg}'], p[avg]))
display(pd.DataFrame(table).round(3))
pd.DataFrame(table).mean().unstack().round(3)

trained               random              
              avg_1  avg_5 avg_20  avg_1  avg_5 avg_20
Indonesia     0.991  0.978  0.902  0.995  0.984  0.889
Malaysia      0.986  0.976  0.920  0.994  0.980  0.895
Philippines   0.988  0.963  0.852  0.995  0.971  0.811
Singapore     0.965  0.952  0.782  0.994  0.967  0.832
Thailand      0.989  0.955  0.869  0.994  0.987  0.908
India         0.938  0.941  0.905  0.994  0.966  0.863

,avg_1,avg_20,avg_5
random,0.994,0.866,0.976
trained,0.976,0.872,0.961


## 2. Is the forecast a straight line on the averages?

**Imitation** is the closest a straight line on the three averages can get to the GRU's own forecasts. **Share explained** is how much of the forecast it matches (1 = all of it).

In [3]:
rows = {}
for market in markets:
    for seed in SEEDS:
        rows[market, seed] = r2(runs[market, seed]['imitation'], runs[market, seed]['gru'])
pd.Series(rows, name='share explained').unstack().round(3)

,0,1,2
India,0.927,0.870,0.931
Indonesia,0.969,0.964,0.969
Malaysia,0.972,0.964,0.949
Philippines,0.903,0.931,0.938
Singapore,0.853,0.856,0.896
Thailand,0.899,0.837,0.860


The weights on the three averages, all test years together: what the GRU's forecasts use (mean over seeds), and what a line fitted to the real target uses.

In [4]:
def weights(p, target):
    return dict(zip(AVERAGES, LinearRegression().fit(p[AVERAGES], p[target]).coef_))

table = {}
for avg in AVERAGES:
    table['GRU', avg] = per_market(lambda p: weights(p, 'gru')[avg])
    table['real target', avg] = per_market(lambda p: weights(p, 'actual')[avg])
pd.DataFrame(table).sort_index(axis=1).round(2)

GRU              real target             
            avg_1 avg_20 avg_5       avg_1 avg_20 avg_5
Indonesia    0.20   0.19  0.35        0.28   0.25  0.33
Malaysia     0.12   0.14  0.36        0.20   0.45  0.24
Philippines  0.17   0.15  0.32        0.21   0.23  0.32
Singapore    0.07   0.12  0.31        0.18   0.28  0.36
Thailand     0.14   0.11  0.37        0.22   0.31  0.37
India        0.08   0.27  0.36        0.21   0.29  0.37

## 3. Does training help?

R2 of the forecasts against the average so far, the same score as step 7, mean over seeds.

- **Imitation:** the GRU's forecast with everything but the straight line thrown away.
- **Untrained GRU + line:** random memory, with only the last line fitted to the target.

In [5]:
table = {
    'HAR': {market: r2_vs_average(har[market]['predicted'], har[market]) for market in markets},
    'GRU': per_market(lambda p: r2_vs_average(p['gru'], p)),
    'imitation': per_market(lambda p: r2_vs_average(p['imitation'], p)),
    'untrained GRU + line': per_market(lambda p: r2_vs_average(p['random_gru_forecast'], p)),
}
pd.DataFrame(table).round(3)

,HAR,GRU,imitation,untrained GRU + line
Indonesia,0.412,0.399,0.400,0.408
Malaysia,0.430,0.349,0.368,0.427
Philippines,0.274,0.245,0.265,0.262
Singapore,0.307,0.217,0.263,0.303
Thailand,0.486,0.433,0.478,0.443
India,0.408,0.379,0.409,0.363


## 4. If we erase an average, does the forecast get worse?

Take out of the memory whatever moves in step with an average, then let the GRU forecast from what's left. **Control** erases a single old day (15 days ago). The forecast scores use the same R2 as above, mean over seeds.

In [6]:
table = {'nothing erased': per_market(lambda p: r2_vs_average(p['gru'], p))}
for name in AVERAGES + ['all', 'control']:
    table[name] = per_market(lambda p: r2_vs_average(p[f'forecast_without_{name}'], p))
pd.DataFrame(table).round(3)

,nothing erased,avg_1,avg_5,avg_20,all,control
Indonesia,0.399,0.124,0.050,0.094,-0.007,0.347
Malaysia,0.349,0.118,-0.005,0.013,-0.039,0.277
Philippines,0.245,0.062,-0.014,0.038,-0.047,0.208
Singapore,0.217,-0.003,-0.088,-0.016,-0.091,0.141
Thailand,0.433,0.048,-0.114,-0.071,-0.173,0.319
India,0.379,0.147,-0.040,-0.019,-0.096,0.311


Check that erasing worked: R2 of a fresh line trying to read the average back from the erased memory, on the test years. Near 0 means it's gone.

In [7]:
pd.DataFrame({name: per_market(lambda p: r2(p[f'readable_after_{name}'], p[name])) for name in AVERAGES}).round(3)

,avg_1,avg_5,avg_20
Indonesia,-0.034,-0.055,-0.072
Malaysia,-0.110,-0.187,-0.242
Philippines,-0.074,-0.157,-0.267
Singapore,-0.041,-0.077,-0.106
Thailand,-0.076,-0.118,-0.153
India,-0.038,-0.062,-0.083
